# DSL blender_arch — Notebook de referencia
Ejecutar dentro de Blender usando el kernel `blender_notebook`  
o en el **Scripting editor** de Blender (celda por celda).

```
Ejes: X = ancho, Y = fondo/profundidad, Z = alto
Unidades: metros
```

## 0 — Setup e imports

In [1]:
import sys, os, bpy

# Añade la carpeta del DSL al path si aún no está
DSL_PATH = os.path.expanduser(
    '~/Desktop/PROGRAMACION_Y_DISEÑO/MISIS/5TO_SEMESTRE/TESIS_1/DSL/blender'
)
if DSL_PATH not in sys.path:
    sys.path.insert(0, DSL_PATH)

import blender_arch as A
print('DSL cargado. Funciones disponibles:', len(A.__all__))

DSL cargado. Funciones disponibles: 33


## 1 — Escena limpia

In [2]:
# Opción A: factory reset (más limpio, borra todo)
bpy.ops.wm.read_factory_settings(use_empty=True)

# Opción B: solo borra objetos y datos huérfanos (preserva settings)
# A.limpiar_escena()

{'FINISHED'}

## 2 — Muro simple

In [3]:
# Muro alineado a +X, origen en (0,0,0)
m = A.crear_muro(
    nombre   = 'Fachada_Sur',
    largo    = 8.0,   # metros en X
    alto     = 3.0,   # metros en Z
    grosor   = 0.25,  # metros en Y
    origen   = (0, 0, 0),
    material = 'Hormigon',
)
print('root:', m.name, '| categoria:', m['arch_categoria'])

root: Fachada_Sur | categoria: muro


In [4]:
# Muro lateral: girar 90° en Z para alinearlo a +Y
A.crear_muro('Muro_Lateral', largo=6.0, alto=3.0, grosor=0.25,
             origen=(0, 0.25, 0), rotacion_z=90)

bpy.data.objects['Muro_Lateral']

## 3 — Vanos: ventanas y puertas

In [5]:
# Ventana con 2 columnas × 1 fila de paños, posicionada en la fachada
v = A.crear_ventana(
    nombre        = 'Ventana_Sala',
    ancho         = 1.5,
    alto          = 1.1,
    prof_marco    = 0.25,   # igual al grosor del muro
    divisiones    = (2, 1), # (nx, ny)
    origen        = (1.5, 0, 0.9),  # X=posición horizontal, Z=altura del alféizar
    material_marco = 'Marco_Blanco',
    material_vidrio = 'Vidrio',
)

# Ventana tipo fijo (sin divisiones)
A.crear_ventana('Ventana_Fija', ancho=0.6, alto=0.6, divisiones=(1,1),
                prof_marco=0.25, origen=(6.0, 0, 1.2))

bpy.data.objects['Ventana_Fija']

In [6]:
# Puerta sencilla en la fachada
p = A.crear_puerta(
    nombre         = 'Puerta_Principal',
    ancho          = 1.0,
    alto           = 2.15,
    prof_marco     = 0.25,
    origen         = (3.5, 0, 0),
    material_panel = 'Madera',
    material_marco = 'Marco_Blanco',
)

In [7]:
# Abrir múltiples vanos en el muro con una sola operación booleana
# centros_world = lista de (x, y, z) del centro de cada hueco en coords globales
A.abrir_vanos_batch_rectangulares(
    muro          = m,   # root o mesh del muro
    centros_world = [(1.0, 0, 1.5), (3.5, 0, 1.5), (6.0, 0, 1.5)],
    ancho         = 1.2,
    alto          = 1.0,
)

bpy.data.objects['Fachada_Sur_Cuerpo']

In [8]:
# Rejilla de vanos (ej. fachada de oficina: 2 filas × 4 columnas)
muro_oficina = A.crear_muro('Fachada_Oficina', largo=10, alto=6, grosor=0.2,
                             origen=(0, 5, 0))
A.abrir_vanos_grid_local(
    muro  = muro_oficina,
    filas = 2, cols = 4,
    x0    = 1.0, z0 = 1.0,  # centro del primer vano en local
    dx    = 2.2, dz = 2.5,  # separación entre centros
    ancho = 1.5, alto = 1.2,
)

bpy.data.objects['Fachada_Oficina_Cuerpo']

## 4 — Habitación completa y composites de alto nivel

In [9]:
# Una sola llamada genera: 4 muros + losa de suelo + losa de techo
hab = A.crear_habitacion(
    nombre          = 'Sala_Estar',
    ancho           = 5.0,   # interior en X
    fondo           = 4.0,   # interior en Y
    alto            = 3.0,   # interior en Z
    grosor_muro     = 0.2,
    grosor_losa     = 0.2,
    con_suelo       = True,
    con_techo       = True,
    origen          = (0, 0, 0),
    material_muro   = 'Muro_Pintura',
    material_suelo  = 'Piso_Madera',
    material_techo  = 'Losacero',
)
print('Hijos del root habitación:', [c.name for c in hab.children])

Hijos del root habitación: ['Sala_Estar_Muro_Este', 'Sala_Estar_Muro_Norte', 'Sala_Estar_Muro_Oeste', 'Sala_Estar_Muro_Sur', 'Sala_Estar_Suelo', 'Sala_Estar_Techo']


### 4b — Casa de N pisos

In [3]:
# Casa de 2 pisos con tejado a dos aguas
casa = A.crear_casa_n_pisos(
    nombre           = 'Casa_Familiar',
    pisos            = 2,
    ancho            = 10.0,
    fondo            = 8.0,
    alto_piso        = 2.80,   # libre + losa (2.60 libre + 0.20 losa)
    grosor_muro      = 0.20,
    grosor_losa      = 0.20,
    con_tejado       = True,
    altura_cumbrera  = 1.80,
    voladizo         = 0.40,
    origen           = (0, 0, 0),
    material_muro    = 'Ladrillo_Rojo',
    material_losa    = 'Hormigon',
    material_tejado  = 'Teja',
)
hijos = [c.name for c in casa.children]
print(f'Casa con {len(hijos)} elementos: {hijos}')

# Casa de 1 piso (cabaña / bungalow)
A.crear_casa_n_pisos(
    'Cabana', pisos=1, ancho=7.0, fondo=6.0, alto_piso=2.60,
    altura_cumbrera=1.20, voladizo=0.50,
    origen=(14, 0, 0),
    material_muro='Adobe', material_tejado='Teja',
)

Casa con 11 elementos: ['Casa_Familiar_P1_Losa', 'Casa_Familiar_P1_Muro_Este', 'Casa_Familiar_P1_Muro_Norte', 'Casa_Familiar_P1_Muro_Oeste', 'Casa_Familiar_P1_Muro_Sur', 'Casa_Familiar_P2_Losa', 'Casa_Familiar_P2_Muro_Este', 'Casa_Familiar_P2_Muro_Norte', 'Casa_Familiar_P2_Muro_Oeste', 'Casa_Familiar_P2_Muro_Sur', 'Casa_Familiar_Tejado']


bpy.data.objects['Cabana']

### 4c — Edificio de N pisos

In [4]:
# Edificio residencial de 6 pisos con ventanas en fachada y columnas internas
edificio = A.crear_edificio_n_pisos(
    nombre              = 'Torre_Residencial',
    pisos               = 6,
    ancho               = 12.0,
    fondo               = 15.0,
    alto_piso           = 2.80,
    grosor_muro         = 0.20,
    grosor_losa         = 0.20,
    con_techo_plano     = True,
    con_columnas        = True,
    seccion_columna     = 'rect',
    dim_columna         = 0.30,
    modulo_columna_x    = 4.0,
    modulo_columna_y    = 5.0,
    ventanas_fachada    = True,
    ventana_ancho       = 1.20,
    ventana_alto        = 1.10,
    ventana_alfeizar    = 0.90,
    ventana_separacion  = 2.40,
    origen              = (0, 20, 0),
    material_fachada    = 'Muro_Pintura_Gris',
    material_losa       = 'Hormigon',
    material_vidrio     = 'Vidrio_Templado',
    material_marco      = 'Marco_Aluminio',
    material_columna    = 'Hormigon',
)
print(f'Edificio: {len(list(edificio.children))} objetos hijos')
print(f'Altura total: {6 * 2.80:.1f} m')

# Edificio de oficinas — 4 pisos, sin columnas, sin ventanas automáticas
A.crear_edificio_n_pisos(
    'Edificio_Oficinas', pisos=4, ancho=20.0, fondo=12.0,
    alto_piso=3.20, con_columnas=False,
    ventanas_fachada=False,   # se añadirán manualmente
    origen=(25, 20, 0),
    material_fachada='Muro_Pintura',
)

Edificio: 79 objetos hijos
Altura total: 16.8 m


bpy.data.objects['Edificio_Oficinas']

### 4a — Habitación simple (4 muros + suelo + techo)

## 5 — Elementos estructurales

In [12]:
# Columna rectangular
A.crear_columna('Col_Rect_1', seccion='rect',
                ancho=0.30, fondo=0.30, alto=3.0,
                origen=(0, 0, 0), material='Hormigon')

# Columna circular (bug corregido: base ahora en Z=0)
A.crear_columna('Col_Circ_1', seccion='circ',
                diametro=0.4, alto=3.5,
                origen=(2, 0, 0), material='Acero')

bpy.data.objects['Col_Circ_1']

In [13]:
# Losa de entrepiso
A.crear_losa_rectangular('Entrepiso_1',
                          ancho=8.0, fondo=6.0, espesor=0.20,
                          origen=(0, 0, 3.0), material='Hormigon')

# Piso de acabado (alias semántico de losa, espesor delgado)
A.crear_piso('Piso_Sala', ancho=5.0, fondo=4.0, espesor=0.02,
             origen=(0.2, 0.2, 0), material='Parquet')

bpy.data.objects['Piso_Sala']

In [14]:
# Techo plano con pendiente hacia +X
A.crear_techo_plano('TechoPlano', ancho=8, fondo=6, espesor=0.2,
                    caida_x=0.1, caida_y=0.0, origen=(0, 0, 3.0))

# Tejado a dos aguas
A.crear_tejado_dos_aguas(
    nombre          = 'Tejado_Casa',
    ancho           = 8.0,
    fondo           = 6.0,
    altura_cumbrera = 2.0,
    espesor         = 0.15,
    voladizo_x      = 0.5,
    voladizo_y      = 0.3,
    origen          = (0, 0, 3.0),
    material        = 'Teja',
)

bpy.data.objects['Tejado_Casa']

## 6 — Escaleras y barandas

In [15]:
# Escalera recta con contrahuellas y zancas laterales
esc = A.crear_escalera_recta(
    nombre             = 'Escalera_Principal',
    huella             = 0.28,   # profundidad del peldaño (m)
    contrahuella       = 0.175,  # altura del escalón  (m)
    ancho              = 1.2,
    num_peldanos       = 16,
    con_contrahuellas  = True,
    con_zancas         = True,
    espesor_zanca      = 0.15,
    origen             = (0, 2, 0),
    material           = 'Hormigon',
)
# Altura total alcanzada:
print(f'Sube {16 * 0.175:.2f} m en {16 * 0.28:.2f} m de planta')

Sube 2.80 m en 4.48 m de planta


In [16]:
# Baranda con 2 travesaños intermedios
A.crear_baranda_lineal(
    nombre         = 'Baranda_Terraza',
    largo          = 8.0,
    altura         = 1.1,
    poste_cada     = 1.0,
    num_travesanos = 2,
    origen         = (0, 6.2, 3.0),
    material       = 'Acero_Inox',
)

bpy.data.objects['Baranda_Terraza']

## 7 — Mobiliario interior

In [17]:
# Mesa de comedor
mesa = A.crear_mesa('Mesa_Comedor', ancho=2.0, fondo=1.0, alto=0.75,
                    origen=(1.0, 1.0, 0))

# 4 sillas alrededor de la mesa usando anclar_a
offsets = [
    ('frente',    (0.5,  -0.55, -0.35)),
    ('detras',    (0.5,   0.55, -0.35)),
    ('izquierda', (-0.6,  0.25, -0.35)),
    ('derecha',   ( 0.1,  0.25, -0.35)),
]
for i, (punto, off) in enumerate(offsets):
    s = A.crear_silla(f'Silla_{i+1}')
    A.anclar_a(s, mesa, punto_anchor=punto, offset=off)

In [18]:
# Sofá
A.crear_sofa('Sofa_3P', ancho=2.2, fondo=0.9, alto_asiento=0.44,
             origen=(0.3, 3.2, 0), material='Tela_Gris')

# Cama matrimonial
A.crear_cama('Cama_Master', ancho=1.6, largo=2.0, alto_cabecero=1.1,
             origen=(8.5, 0.5, 0), material_colchon='Textil_Blanco')

# Estantería / librero
A.crear_estanteria('Librero', ancho=1.2, alto=2.1, fondo=0.3,
                   num_estantes=4, con_fondo=True,
                   origen=(5.5, 0.2, 0), material='Madera_Roble')

# Armario / closet con 3 puertas
A.crear_armario('Closet_Dorm', ancho=2.4, alto=2.4, fondo=0.6,
                num_puertas=3, con_zocalo=True,
                origen=(8.0, 3.5, 0))

bpy.data.objects['Closet_Dorm']

## 8 — Exterior: terreno y vegetación

In [19]:
# Terreno del sitio
A.crear_terreno_plano('Sitio', ancho=24, fondo=18, espesor=0.3,
                      origen=(-2, -2, -0.3))

# Árboles en el jardín
for i, (x, y, r, h) in enumerate([
    (-1.5, 1.0, 1.8, 3.0),
    (-1.5, 5.5, 1.5, 2.5),
    ( 9.5, 3.0, 2.0, 3.5),
]):
    A.crear_arbol_simple(f'Arbol_{i+1}',
                         radio_copa=r, altura_copa=h,
                         altura_tronco=1.5,
                         origen=(x, y, 0))

## 9 — Geometría genérica

In [20]:
from math import cos, sin, pi

# Extrusión de perfil libre (columna hexagonal)
hex_pts = [(cos(i * pi/3) * 0.15, sin(i * pi/3) * 0.15) for i in range(6)]
A.extruir_perfil('Col_Hex', puntos_2d=hex_pts, altura=3.0,
                 material='Concreto').location = (12, 0, 0)

# Extrusión con hueco interior (marco de ventana en planta)
A.extruir_con_huecos(
    nombre   = 'Marco_Estructural',
    contorno = [(0,0),(2,0),(2,1.5),(0,1.5)],
    agujeros = [[(0.15,0.15),(1.85,0.15),(1.85,1.35),(0.15,1.35)]],
    altura   = 0.12,
    material = 'Metal',
).location = (12, 2, 0)

# Tubería (instalaciones)
A.crear_tuberia(
    'Tuberia_Agua',
    puntos_3d = [(0,0,0.3),(0,4,0.3),(4,4,0.3),(4,4,2.5)],
    radio     = 0.025,
    material  = 'Cobre',
    origen    = (0.1, 0.1, 0),
)

bpy.data.objects['Tuberia_Agua']

## 10 — Iluminación y cámara

In [21]:
# Luz solar
A.agregar_luz('Sol', tipo='SUN', ubicacion=(5, -5, 10),
              energia=5.0, color=(1.0, 0.97, 0.88))

# Luces de techo (área) — una por habitación
A.agregar_luz('Luz_Sala',  tipo='AREA', ubicacion=(2.5, 2.0, 2.8),
              energia=400, color=(1.0, 0.95, 0.85))
A.agregar_luz('Luz_Dorm',  tipo='AREA', ubicacion=(9.5, 2.0, 2.8),
              energia=300, color=(1.0, 0.93, 0.80))

# Spot de entrada
A.agregar_luz('Spot_Entrada', tipo='SPOT', ubicacion=(4.0, -1.5, 3.5),
              energia=800, angulo_spot=35.0)

bpy.data.objects['Spot_Entrada']

In [22]:
# Cámara principal — vista isométrica
cam_iso = A.crear_camara(
    nombre        = 'Cam_Isometrica',
    ubicacion     = (10, -12, 8),
    rotacion      = (60, 0, 45),   # grados (rx, ry, rz)
    focal_length  = 35,
    activa        = True,
)

# Cámara ortogonal — alzado frontal
A.crear_camara(
    nombre    = 'Cam_Alzado',
    ubicacion = (4, -15, 1.5),
    rotacion  = (90, 0, 0),
    tipo      = 'ORTHO',
    activa    = False,
)

bpy.data.objects['Cam_Alzado']

## 11 — Validación de mallas

In [5]:
# Valida todos los objetos MESH de la escena
import blender_arch as A

resultados = {}
for obj in bpy.context.scene.objects:
    if obj.type == 'MESH':
        rep = A.validar_malla(obj, verbose=False)
        if not rep['ok']:
            resultados[obj.name] = rep

if resultados:
    print('Mallas con problemas:')
    for nombre, rep in resultados.items():
        print(f'  {nombre}: {rep}')
else:
    print('✓ Todas las mallas están limpias')

✓ Todas las mallas están limpias


## 12 — Exportar

In [6]:
# Guardar como .blend
A.exportar_escena('/tmp/mi_proyecto.blend', formato='BLEND')

# Exportar a GLTF (útil para web / Three.js)
# A.exportar_escena('/tmp/mi_proyecto.glb', formato='GLTF')

# Exportar a OBJ
# A.exportar_escena('/tmp/mi_proyecto.obj', formato='OBJ')

Info: Saved as "mi_proyecto.blend"
[DSL] Escena exportada → /tmp/mi_proyecto.blend


## Cheatsheet rápida

```python
import blender_arch as A

# --- Estructurales ---
A.crear_muro('M', largo=4, alto=3, grosor=0.2, origen=(0,0,0))
A.crear_columna('C', seccion='circ', diametro=0.3, alto=3, origen=(2,0,0))
A.crear_losa_rectangular('L', ancho=4, fondo=3, espesor=0.2, origen=(0,0,3))
A.crear_piso('P', ancho=4, fondo=3, espesor=0.02, material='Parquet')
A.crear_techo_plano('T', ancho=4, fondo=3, espesor=0.2, origen=(0,0,3))
A.crear_tejado_dos_aguas('R', ancho=8, fondo=6, altura_cumbrera=2, origen=(0,0,3))

# --- Huecos ---
A.crear_ventana('V', ancho=1.2, alto=1.0, divisiones=(2,1), origen=(1,0,0.9))
A.crear_puerta('P', ancho=0.9, alto=2.1, origen=(3,0,0))
A.abrir_vanos_batch_rectangulares(muro, [(x,y,z),...], ancho=1.2, alto=1.0)
A.abrir_vanos_grid_local(muro, filas=2, cols=4, x0=1, z0=0.8, dx=2, dz=2)

# --- Circulación ---
A.crear_escalera_recta('E', huella=0.28, contrahuella=0.175, ancho=1.2, num_peldanos=14)
A.crear_baranda_lineal('B', largo=4, altura=1.05, poste_cada=1.0, num_travesanos=2)

# --- Mobiliario ---
A.crear_mesa('M', ancho=1.6, fondo=0.8, alto=0.75)
A.crear_silla('S')
A.crear_sofa('SF', ancho=2.2)
A.crear_cama('CM', ancho=1.6, largo=2.0)
A.crear_estanteria('EST', ancho=1.0, alto=2.1, num_estantes=4)
A.crear_armario('ARM', ancho=2.0, alto=2.4, num_puertas=2)

# --- Exterior ---
A.crear_terreno_plano('T', ancho=20, fondo=15, origen=(0,0,-0.3))
A.crear_arbol_simple('A', radio_copa=1.5, altura_tronco=1.5, origen=(5,3,0))

# --- Composites ---
A.crear_habitacion('H', ancho=5, fondo=4, alto=3)
A.crear_casa_n_pisos('Casa', pisos=2, ancho=10, fondo=8, alto_piso=2.8,
                     con_tejado=True, material_muro='Ladrillo_Rojo')
A.crear_edificio_n_pisos('Edif', pisos=6, ancho=12, fondo=15, alto_piso=2.8,
                          con_columnas=True, ventanas_fachada=True)

# --- Geometría libre ---
A.extruir_perfil('E', puntos_2d=[(x,y),...], altura=3.0)
A.extruir_con_huecos('EH', contorno=[...], agujeros=[[...]])
A.crear_tuberia('TB', puntos_3d=[(x,y,z),...], radio=0.025)

# --- Luz / Cámara ---
A.agregar_luz('L', tipo='AREA', ubicacion=(2,2,3), energia=400)
A.crear_camara('C', ubicacion=(8,-10,6), rotacion=(60,0,45))

# --- Utilidades ---
A.anclar_a(silla, mesa, punto_anchor='frente', offset=(0.5, -0.5, 0))
A.limpiar_escena()
A.exportar_escena('/tmp/arch.blend')   # BLEND / OBJ / FBX / GLTF / STL
A.validar_malla(obj)
```